# Day 6 — Cost, Streaming & Async

---

Three production concerns before the capstone:

1. **Cost control** — count tokens, cap spend, avoid surprises.
2. **Streaming** — make replies feel instant.
3. **Async** — send many AI requests in parallel without waiting.

We'll use Together AI throughout.

In [ ]:
!pip install together tiktoken python-dotenv --quiet

In [3]:
import os, time, json, tiktoken, asyncio
from dotenv import load_dotenv
from together import Together
from openai import AsyncOpenAI

load_dotenv()
assert os.getenv("TOGETHER_API_KEY"), "Set TOGETHER_API_KEY in .env"

client = Together()
MODEL = "openai/gpt-oss-20b"
enc = tiktoken.get_encoding("cl100k_base")

## 1. Count tokens *before* you spend

Every provider bills per token. `tiktoken` counts them locally — free, instant, exact for OpenAI models and close enough for Together AI.

**Rough Together AI price:** ~$0.18 per million tokens (input **and** output) for LLaMA 8B.

In [4]:
def estimate_cost(prompt: str, max_output_tokens: int = 500,
                  price_in_per_M: float = 0.18, price_out_per_M: float = 0.18):
    in_tokens = len(enc.encode(prompt))
    cost = (in_tokens / 1_000_000) * price_in_per_M + \
           (max_output_tokens / 1_000_000) * price_out_per_M
    return in_tokens, cost

p = "Explain quantum tunneling in simple terms." * 5
tokens, cost = estimate_cost(p)
print(f"input tokens        : {tokens}")
print(f"worst-case cost     : ${cost:.6f}")
print(f"1,000 requests cost : ${cost*1000:.4f}")

input tokens        : 41
worst-case cost     : $0.000097
1,000 requests cost : $0.0974


**Three habits that save real money:**

1. **Cap `max_tokens`.** Prevents the model from writing a novel.
2. **Trim the system prompt.** 100 tokens saved × 1M requests = real money.
3. **Log every call.** You can't fix what you can't see.

## 2. Streaming — same time, feels 10× faster

Without streaming, the user waits 4–8 seconds for the whole reply. 
With streaming, the first word appears in ~300 ms. **Same total time.** But the experience is completely different.

How: pass `stream=True` and loop the chunks.

In [13]:
t0 = time.time()
first = None

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "List 3 unusual uses for a paperclip."}],
    stream=True,
    max_tokens=300,
)

for chunk in stream:
    if not getattr(chunk, "choices", None):
        continue

    choice = chunk.choices[0]
    delta = None

    if getattr(choice, "delta", None) is not None:
        if isinstance(choice.delta, dict):
            delta = choice.delta.get("content")
        else:
            delta = getattr(choice.delta, "content", None)

    if delta is None and getattr(choice, "message", None) is not None:
        if isinstance(choice.message, dict):
            delta = choice.message.get("content")
        else:
            delta = getattr(choice.message, "content", None)

    if delta:
        if first is None:
            first = time.time() - t0
        print(delta, end="", flush=True)

print(f"\n\nFirst token in {first:.2f}s | total {time.time()-t0:.2f}s")

**Three Unusual Uses for a Paperclip**

| # | Unusual Use | Quick Explanation |
|---|-------------|-------------------|
| 1 | **Emergency lock pick** | Straighten a paperclip and use it to pick the pin tumbler of a simple lock (e.g., a small padlock). Works best on single‑pin or basic cartridge locks. |
| 2 | **Miniature piano wire tensioner** | Hook a paperclip to a string or light wire to adjust tension when building a small homemade piano or string instrument. The clip’s flexibility allows fine‑tuning. |
| 3 | **Portable bookmark for magnetic books** | Attach a paperclip to a stiff cardboard or plastic backing, then stick it to the page of a magnet‑based e‑reader or a book with a steel spine. It keeps the page in place without tearing. |

These uses showcase how a plain office supply can be repurposed into handy tools beyond its standard office role.

First token in 0.98s | total 2.29s


You just saw tokens print live instead of appearing all at once. That's the single biggest UX difference between a good AI app and a mediocre one.

## 3. Async — send many requests in parallel

### What is async, in plain English?

A regular Python program does one thing at a time. If you make 5 AI calls, they happen one after another — total time = 5 × per-call time.

**Async** lets Python *start* all 5 calls, then wait for them together. Total time = ~1 × per-call time (as fast as the slowest one).

Why this works: LLM calls are 99% *waiting* on the network. Async is the perfect fit for anything that spends most of its time waiting.

### The syntax

Three new keywords:
- `async def` — defines an async function.
- `await` — waits for one async call to finish.
- `asyncio.gather(...)` — starts many at once and waits for all.

In [ ]:
# Together AI's SDK is OpenAI-compatible, so we use AsyncOpenAI
# pointed at Together's base URL.
aio = AsyncOpenAI(
    api_key=os.getenv("TOGETHER_API_KEY"),
    base_url="https://api.together.xyz/v1",
)

async def ask(prompt):
    r = await aio.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=60,
    )

    print("DEBUG: raw response:", r)

    choice = None
    if getattr(r, "choices", None):
        choice = r.choices[0]
    elif isinstance(r, dict):
        choice = r.get("choices", [None])[0]

    if choice is None:
        print("DEBUG: no choice object found")
        raise ValueError("No choice object found in model response")

    content = None
    if getattr(choice, "message", None) is not None:
        msg = getattr(choice, "message")
        if isinstance(msg, dict):
            content = msg.get("content")
        else:
            content = getattr(msg, "content", None)

    if content is None and isinstance(choice, dict):
        content = choice.get("message", {}).get("content") or choice.get("text")

    if content is None:
        content = getattr(choice, "text", None)

    if content is None:
        delta = getattr(choice, "delta", None)
        if isinstance(delta, dict):
            content = delta.get("content")
        else:
            content = getattr(delta, "content", None) if delta is not None else None

    if content is None:
        content = getattr(choice, "content", None)

    if content is None:
        print("DEBUG: choice object:", choice)
        raise ValueError("Could not extract text from the model response")

    return content.strip()

# Try one call
result = await ask("Say hi in Japanese.")
print("result:", repr(result))
print(result)

こんにちは！


### The magic: `asyncio.gather` — parallel calls

Send 4 questions to Together AI at the same time. In a Jupyter cell we can `await` at the top level.

In [16]:
PROMPTS = [
    "1-line joke about databases",
    "1-line joke about Python",
    "1-line joke about AI",
    "1-line joke about coffee",
]

t0 = time.time()
results = await asyncio.gather(*(ask(p) for p in PROMPTS))
print(f"parallel : {time.time()-t0:.2f}s for {len(PROMPTS)} calls")
for r in results:
    print(" -", r)

parallel : 1.88s for 4 calls
 - 
 - 
 - Why did the AI break up with its spreadsheet? Because it couldn't find a solid state of mind
 - 


Compare that to running them one-by-one — you'd wait for the total. Parallel is roughly the same time as *one* call, no matter how many you fan out.

**When to reach for async:**
- Multiple LLM calls per user request (classify → then answer, retrieve → then generate).
- Fanning out one question to several models to compare.
- Any FastAPI endpoint — `async def` keeps the server responsive.

## Recap

- **Cost:** count tokens with `tiktoken`, cap output with `max_tokens`, log every call.
- **Streaming:** `stream=True` → first token appears in ~300 ms. UX transformation.
- **Async:** `async def` + `await` + `asyncio.gather` = many AI calls in the time of one.
- Together AI's async client is `AsyncOpenAI(api_key=..., base_url="https://api.together.xyz/v1")`.

Tomorrow: assemble everything into a production-shaped AI Assistant.